In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
%matplotlib inline
import seaborn as sns

In [ ]:
df=pd.read_csv('Amazon Sale Report.csv',encoding= 'unicode_escape')

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
#drop unrelated/blank columns
df.drop(['New','PendingS'], axis=1, inplace=True)

In [ ]:
df.info()

In [ ]:
pd.isnull(df) 
# checking null value 

In [ ]:
pd.isnull(df).sum()
# sum will give total values of null values

In [ ]:
df.shape

In [ ]:
# Avoid dropping every row that has any missing value: several columns are optional (for example, promotion IDs and fulfillment details).
# Keep the source rows and handle missing values only where a specific analysis requires them.
df['ship-postal-code'] = pd.to_numeric(df['ship-postal-code'], errors='coerce').astype('Int64')
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
df['Qty'] = pd.to_numeric(df['Qty'], errors='coerce')

# Report the remaining missing values so their impact is visible.
df.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
# Postal codes are identifiers, not quantities. Nullable integer dtype preserves missing values.
df['ship-postal-code'] = pd.to_numeric(df['ship-postal-code'], errors='coerce').astype('Int64')

In [ ]:
#checking whether the  data type change or not
df['ship-postal-code'].dtype

In [ ]:
# Dataset dates are in MM-DD-YY format. Invalid values become NaT and can be inspected.
df['Date'] = pd.to_datetime(df['Date'], format='%m-%d-%y', errors='coerce')
print('Rows with invalid/missing dates:', df['Date'].isna().sum())

In [ ]:
df.columns

In [ ]:
# Persist the column rename so later analysis uses a consistent, descriptive name.
df.rename(columns={'Qty': 'Quantity'}, inplace=True)
df[['Quantity', 'Amount']].head()

In [ ]:
#describe() method return description of the data in the DataFrame(i.e count,mean,std,min..etc)
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
#use describe() for specific columns
df[['Quantity','Amount']].describe()

# Exploratory Data Analysis


## Business questions

- Which sizes and categories appear most often, and do order-line counts differ from total quantity?
- How are records distributed by fulfillment and courier status?
- What share of records are marked B2B versus retail?
- Which shipping states contribute the most order records?
- How does order amount vary across frequently occurring categories?

## Interpretation notes

This file is an order-level/line-level exploratory analysis, not a customer-level cohort analysis. Check cancellation/return status and duplicate Order IDs before interpreting counts as delivered orders. Missing optional fields are retained where possible; missingness should be assessed per question rather than dropping every row indiscriminately.

In [ ]:
df.columns

### size

In [ ]:
# The next cell displays the same distribution with bar labels.
df['Size'].value_counts(dropna=False)

In [ ]:
ax=sns.countplot(x='Size' ,data=df)

for bars in ax.containers:
    ax.bar_label(bars)

#### Note: From above Graph you can see that most of the people buys M-Size 

# Group By
#### The groupby() function in pandas is used to group data based on one or more columns in a DataFrame

In [ ]:
df.groupby(['Size'], as_index=False)['Quantity'].sum().sort_values(by='Quantity',ascending=False)

In [ ]:
S_Qty=df.groupby(['Size'], as_index=False)['Quantity'].sum().sort_values(by='Quantity',ascending=False)

sns.barplot(x='Size',y='Quantity', data=S_Qty)


##### Note: From above Graph you can see that most of the Qty buys M-Size in the sales 

### Courier Status 

In [ ]:
sns.countplot(data=df, x='Courier Status',hue= 'Status')

In [ ]:
plt.figure(figsize=(10,5))

ax=sns.countplot(data=df, x='Courier Status',hue= 'Status')

plt.show()

##### Note: From above Graph the majority of the orders are shipped through the courier.

In [ ]:
# Amount is numeric, so a histogram is appropriate here.
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='Amount', bins=40, kde=True)
plt.title('Order Amount Distribution')
plt.xlabel('Order amount')
plt.ylabel('Order-line count')
plt.tight_layout()
plt.show()

In [ ]:
# Category is categorical: compare counts with a bar chart, not a histogram.
top_categories = df['Category'].value_counts().head(10).sort_values()
plt.figure(figsize=(10, 6))
top_categories.plot(kind='barh')
plt.title('Top 10 Categories by Record Count')
plt.xlabel('Order-line count')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

##### Note: From above Graph you can see that most of the  buyers are T-shirt 

In [ ]:
# Check the mix of B2B and retail records as a table before plotting.
B2B_Check = df['B2B'].value_counts(dropna=False)
B2B_Check.rename(index={True: 'B2B', False: 'Retail'}).to_frame('order_line_count')

In [ ]:
# Checking B2B Data  by using pie chart 
B2B_Check = df['B2B'].value_counts()

#  Plot the pie chart
plt.pie(B2B_Check, labels=B2B_Check.index, autopct='%1.1f%%')
#plt.axis('equal')
plt.show()

##### Note : From above chart  we can see that maximum i.e. 99.3% of buyers are retailers and 0.7% are B2B buyers

In [ ]:
#  Prepare data for pie chart
a1 = df['Fulfilment'].value_counts()

# Step 4: Plot the pie chart
fig, ax = plt.subplots()

ax.pie(a1, labels=a1.index, autopct='%1.1f%%', radius=0.7, wedgeprops=dict(width=0.6))
ax.set(aspect="equal")

plt.show()


##### Note: From above chart you can see that most of the  Fulfilment  are  amazon 

In [ ]:
# Compare order amount across the five most frequent categories.
top_category_names = df['Category'].value_counts().head(5).index
plot_df = df[df['Category'].isin(top_category_names)].copy()
plt.figure(figsize=(12, 6))
sns.boxplot(data=plot_df, x='Category', y='Amount')
plt.title('Order Amount by Top 5 Categories')
plt.xlabel('Category')
plt.ylabel('Order amount')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:


# Plot count of cities by state
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='ship-state')
plt.xlabel('ship-state')
plt.ylabel('count')
plt.title('Distribution of State')
plt.xticks(rotation=90)
plt.show()


In [ ]:
# top_10_States 
top_10_state = df['ship-state'].value_counts().head(10)
# Plot count of cities by state
plt.figure(figsize=(12, 6))
sns.countplot(data=df[df['ship-state'].isin(top_10_state.index)], x='ship-state')
plt.xlabel('ship-state')
plt.ylabel('count')
plt.title('Distribution of  State')
plt.xticks(rotation=45)
plt.show()

#### Note: From above Graph you can see that most of the  buyers are Maharashtra state

### Conclusion

### Conclusion

The notebook explores order-line patterns by size, category, fulfillment, B2B status and shipping state. These charts describe the records in this dataset; they do not necessarily represent unique customers or completed sales. Before making operational decisions, reconcile order-line counts with unique Order IDs and distinguish cancelled, shipped and returned statuses. The category, size and state rankings should be reproduced after rerunning the notebook with the current source file.